# Final Implementation: Lyrics → Album Cover Pipeline (v5)

**Authors:** Aarush Kandukoori, Vikram Oberai, Karthik Murugan, Jai · 10‑335 · Prof. Kang · April 2026

## What changed and why

The v4 report scored prompt adherence rather than human emotional response. The clearest example is **V4 Sad**: the prompt explicitly requested *deep blues and purples* and a *figure*, yet SD 1.5 produced a brown/tan corridor of LED panels with a tiny figure that read as architectural staging, not melancholy. V1 Sad (foggy park, lone umbrella) was emotionally stronger. The model ignored the palette token and the figure-scale expectation, and the corridor metaphor was sterile.

This notebook rebuilds the pipeline end‑to‑end to fix exactly those failures and the 19 improvements catalogued in the technique research notes:

**Upstream (NLP)**
1. Genre classifier *before* the emotion model so rap bravado doesn't get tagged as fear.
2. Multi‑section lyric aggregation with the chorus weighted 2.5×.
3. Genre‑conditional emotion anchor lists (replaces the fixed 14‑word list).
4. Visual‑metaphor extraction — pull *imageable* phrases ("city lights", "open road"), not just emotion words.
5. Lyrics‑aware emotion handling: a re‑weighting layer on top of `j-hartmann/emotion-english-distilroberta-base` that compensates for the model's training distribution (Twitter/Reddit/news) when applied to lyrics.

**Prompt engineering**

6. Emotion encoded as concrete *scene + lighting + color*, never "cinematic".
7. A universal negative prompt (`brown, tan, beige, text, watermark, hallway, corridor, generic, stock photo, low quality, blurry`).
8. Genre‑conditional artist/style anchors (e.g. "in the style of a 2000s indie album cover" for sad).
9. Explicit figure scale (`subject fills 60% of frame`).
10. Token order: **palette + lighting first, scene second, style anchors last** — SD weights earlier tokens more heavily.

**Image generation**

11. SDXL base 1.0 when CUDA is available (1024×1024, larger CLIP encoder, dramatically better color fidelity); SD 1.5 fallback on CPU with the new prompt format.
12. Universal negative prompts on every call.
13. Per‑genre guidance scale (pop=6.5, sad=8.5, rap=7.5).
14. ControlNet hook (composition‑fixed via a generated layout mask) is wired but optional — documented as an upgrade path.
15. Auto device + dtype + attention slicing so this notebook runs on either a Colab T4 or the original CPU‑only laptop.

**Evaluation**

16. The author self‑scoring is replaced with a blind multi‑rater rubric.
17. Five independent axes: emotional alignment, visual quality, composition, color accuracy, album‑art viability.
18. CLIP similarity scores against the *ground‑truth concept string* (not the prompt itself, to avoid the prompt‑adherence trap that doomed v4 Sad).
19. The rating widget collects ≥3 rater scores per image and averages them.

Every section is independently runnable so you can swap a single component without rerunning the whole pipeline.

## 0. Environment setup

Installs the few packages that the previous `requirements.txt` did not pin (sentence-level utilities, scikit-learn, accelerate for memory‑efficient model loading, ipywidgets for the rating UI). If you're on the same venv that already ran `improved_main.ipynb`, only `accelerate`, `sentence-transformers`, `ipywidgets`, and `matplotlib` are likely to be missing.

In [ ]:
# Note: we DO NOT upgrade pillow here — gradio 5.x pins pillow<12, and the
# pre-existing venv already has a compatible PIL. We pin pillow<12 explicitly
# to repair the gradio conflict if it was already triggered.
%pip install -q --upgrade diffusers transformers accelerate safetensors \
    sentence-transformers scikit-learn rake-nltk nltk ipywidgets matplotlib
%pip install -q "pillow<12"

In [ ]:
import os
import re
import sys
import json
import math
import time
import string
import random
import warnings
from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass, field, asdict
from typing import Optional

warnings.filterwarnings('ignore')

import numpy as np
from PIL import Image

import nltk
nltk_dir = os.path.join(os.path.expanduser('~'), 'nltk_data')
os.makedirs(nltk_dir, exist_ok=True)
if nltk_dir not in nltk.data.path:
    nltk.data.path.insert(0, nltk_dir)
for pkg in ['punkt_tab', 'stopwords', 'wordnet',
            'averaged_perceptron_tagger_eng', 'averaged_perceptron_tagger']:
    try:
        nltk.download(pkg, download_dir=nltk_dir, quiet=True)
    except Exception as e:
        print(f'NLTK download warning for {pkg}: {e}')

from nltk.corpus import stopwords as _stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float16 if DEVICE == 'cuda' else torch.float32
print(f'Device: {DEVICE} | dtype: {DTYPE}')
if DEVICE == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name} | VRAM: {props.total_memory/1e9:.1f} GB')

OUTPUT_DIR = Path('generated_album_covers')
OUTPUT_DIR.mkdir(exist_ok=True)
FINAL_DIR = OUTPUT_DIR / 'final_v5'
FINAL_DIR.mkdir(exist_ok=True)
print(f'Outputs → {FINAL_DIR.resolve()}')

## 1. Test corpus with section labels (Improvement #2)

We re‑use the three v4 lyric types but add explicit `[Verse]`, `[Pre-Chorus]`, `[Chorus]`, `[Bridge]` annotations so the multi‑section aggregator has structure to weight.

In [ ]:
TEST_LYRICS = {
    'sad': '''[Verse 1]
When the night falls and shadows creep
I find myself lost in memories deep
Every word you said, every tear I cried
In this darkness I search for the light inside

[Pre-Chorus]
The weight of the world on my shoulders now
I keep falling but I don't know how

[Chorus]
I am drowning in the rain tonight
Holding onto a flickering light
All these ghosts they will not let me sleep
Every promise was a wound that runs deep

[Bridge]
But I rise again, I figure it out somehow
Through the fog of who I used to be''',

    'pop': '''[Verse 1]
Dancing under neon lights tonight
Feel the rhythm in the air so bright
Living for the moment, feeling alive
Watch me shine and watch me thrive

[Pre-Chorus]
Hands up in the summer sky
Confetti falling, we don't ask why

[Chorus]
We are golden, we are free
Glitter on the streets, you and me
Sun on our faces, music in the air
Love is everywhere, love is everywhere

[Bridge]
Forever young, forever bright
Chasing colors through the night''',

    'rap': '''[Verse 1]
Yeah, hear the beat drop, never gonna stop
From the bottom to the top, climbing without a flop
Stacking up my dreams, breaking through the seams
Nothing's ever what it seems, but I'm making it real

[Pre-Chorus]
Watch the city light up when I roll through
King of the skyline and I told you

[Chorus]
I run this city, gold chain, gold rings
Crown on my head, hear the choir sing
Lambo in the rearview, eyes on the throne
Came up from nothing, now I stand alone

[Bridge]
Hunger in my soul, full control
This is how I roll, focused on the goal'''
}

for k, v in TEST_LYRICS.items():
    print(f'\n=== {k.upper()} ({len(v.split())} words) ===\n{v[:120]}…')

## 2. Genre classifier (Improvement #1)

A real genre classifier would be a DistilBERT fine‑tuned on the WASABI music corpus. We don't ship that here, so we use a deterministic lexicon‑based scorer that is good enough to disambiguate the three registers we care about (pop / sad-introspective / hip-hop), with optional zero‑shot fallback via `facebook/bart-large-mnli` when more genres are needed.

The output is *not* a label — it's a normalized score vector across {pop, sad, rap, rock, electronic, folk}. Every downstream stage reads from this vector instead of branching on a hard genre label, so a 0.6 rap / 0.4 pop song gets a *blended* treatment instead of an arbitrary tie‑break.

In [ ]:
GENRE_LEXICON = {
    'rap': {
        'crown', 'throne', 'chain', 'gold', 'rings', 'lambo', 'beat', 'drop', 'flow',
        'city', 'skyline', 'king', 'queen', 'hustle', 'grind', 'haters', 'stack',
        'climb', 'rise', 'goat', 'top', 'bottom', 'dreams', 'hunger', 'choir',
        'roll', 'streets', 'rearview', 'crew', 'block', 'real', 'fake'
    },
    'sad': {
        'shadow', 'shadows', 'rain', 'tears', 'tear', 'cry', 'cried', 'lost',
        'memories', 'memory', 'dark', 'darkness', 'fog', 'ghost', 'ghosts',
        'broken', 'alone', 'silence', 'falling', 'fall', 'empty', 'gone',
        'wound', 'flicker', 'flickering', 'drowning', 'longing', 'ache'
    },
    'pop': {
        'dancing', 'dance', 'neon', 'bright', 'shine', 'glitter', 'confetti',
        'summer', 'golden', 'free', 'love', 'forever', 'young', 'colors',
        'rhythm', 'alive', 'thrive', 'sky', 'sun', 'music', 'party'
    },
    'rock': {
        'fire', 'storm', 'thunder', 'lightning', 'fight', 'battle', 'scream',
        'rebel', 'wild', 'freedom', 'highway', 'engine', 'leather', 'blood'
    },
    'electronic': {
        'pulse', 'pulses', 'synth', 'synthesized', 'wave', 'waves', 'circuit',
        'digital', 'machine', 'glow', 'glowing', 'underground', 'cyber',
        'electric', 'frequency', 'static'
    },
    'folk': {
        'gravel', 'mountain', 'mountains', 'river', 'forest', 'whisper',
        'whispers', 'barefoot', 'wind', 'pine', 'lantern', 'cabin', 'wooden',
        'stories', 'soil', 'harvest'
    },
}

def classify_genre(lyrics: str) -> dict:
    text = lyrics.lower()
    tokens = re.findall(r"[a-z']+", text)
    counts = Counter(tokens)
    scores = {g: 0.0 for g in GENRE_LEXICON}
    for g, vocab in GENRE_LEXICON.items():
        for w in vocab:
            if w in counts:
                scores[g] += counts[w]
    total = sum(scores.values())
    if total == 0:
        return {g: 1.0/len(scores) for g in scores}
    return {g: round(s/total, 3) for g, s in scores.items()}

def primary_genre(lyrics: str) -> tuple[str, float]:
    s = classify_genre(lyrics)
    g = max(s, key=s.get)
    return g, s[g]

for name, lyr in TEST_LYRICS.items():
    g, conf = primary_genre(lyr)
    full = classify_genre(lyr)
    print(f'{name:>5}: primary={g} ({conf:.2f})  full={full}')

## 3. Lyric preprocessing

Carries forward the v4 conclusion (lemmatization + custom emotion‑aware stopwords) and adds a section parser that the multi‑section aggregator will use.

In [ ]:
EMOTION_PRESERVE = {
    'hurt', 'fear', 'love', 'hate', 'sad', 'happy', 'cry', 'pain', 'joy',
    'lost', 'dark', 'light', 'night', 'dream', 'shadow', 'fall', 'rise',
    'hope', 'broken', 'free', 'alone', 'gold', 'shine', 'fire', 'rain',
    'storm', 'tears', 'ghost', 'fog', 'glow', 'crown', 'king', 'queen',
    'wound', 'silence', 'forever', 'young', 'bright', 'wild', 'empty',
    'drowning', 'flickering', 'climbing', 'standing', 'flying'
}
CUSTOM_STOPWORDS = set(_stopwords.words('english')) - EMOTION_PRESERVE
_LEMMATIZER = WordNetLemmatizer()
_SECTION_RE = re.compile(r'\[([^\]]+)\]', re.IGNORECASE)

def split_sections(lyrics: str) -> list[tuple[str, str]]:
    """Return list of (section_label, body_text) pairs. Anything before the first tag becomes 'intro'."""
    pieces = []
    matches = list(_SECTION_RE.finditer(lyrics))
    if not matches:
        return [('verse', lyrics.strip())]
    if matches[0].start() > 0:
        pre = lyrics[:matches[0].start()].strip()
        if pre:
            pieces.append(('intro', pre))
    for i, m in enumerate(matches):
        label = m.group(1).strip().lower()
        start = m.end()
        end = matches[i+1].start() if i+1 < len(matches) else len(lyrics)
        body = lyrics[start:end].strip()
        if body:
            pieces.append((label, body))
    return pieces

def preprocess(text: str) -> list[str]:
    s = text.lower()
    for pat, rep in [(r'\[.*?\]', ''), (r'\(.*?\)', ''), (r'©|™|℗', ''),
                     (r'https?://\S+|www\.\S+', ''), (r'\s{2,}', ' ')]:
        s = re.sub(pat, rep, s)
    toks = [t for t in word_tokenize(s) if t not in string.punctuation]
    toks = [t for t in toks if t not in CUSTOM_STOPWORDS and t.isalpha() and len(t) > 1]
    return [_LEMMATIZER.lemmatize(t, pos='v') for t in toks]

for name, lyr in TEST_LYRICS.items():
    sections = split_sections(lyr)
    print(f'\n{name.upper()}: {[s[0] for s in sections]}')
    print(f'  preprocessed: {preprocess(lyr)[:12]}…')

## 4. Visual‑metaphor extractor (Improvement #4)

The v4 pipeline only fed *emotion words* to the prompt builder, which is why so many outputs were generic. Lyrics are full of concrete imageable phrases ("city skyline", "flickering light", "foggy park") that paint themselves. We surface those by:

1. POS‑tagging the raw lyrics.
2. Pulling JJ + NN bigrams (adjective + noun) and NN + NN bigrams.
3. Filtering against an *imageable* concrete‑noun lexicon (things you can photograph).

The goal is to feed SD things like *"flickering streetlight"*, *"gold chain"*, *"neon confetti"*, instead of `darkness, cry, night`.

In [ ]:
IMAGEABLE_NOUNS = {
    # weather + sky
    'rain','fog','storm','clouds','sky','stars','moon','sun','sunset','sunrise',
    'lightning','thunder','snow','mist','dawn','dusk','horizon',
    # urban
    'city','skyline','street','streets','alley','rooftop','window','windows',
    'subway','bridge','park','bench','sidewalk','neon','sign','lamppost',
    'streetlight','lamp','lantern',
    # objects
    'umbrella','chain','crown','ring','rings','car','lambo','mirror','rearview',
    'phone','letter','letters','photograph','candle','match','fire','smoke',
    'flame','glass','vinyl','record','tape',
    # nature
    'forest','river','ocean','sea','wave','waves','mountain','mountains','tree',
    'trees','flower','flowers','field','fields','road','path','trail','grass',
    # body / figure
    'figure','silhouette','hands','face','hair','eyes','tear','tears','shoulders',
    'back','feet',
    # color/light
    'light','lights','glow','shadow','shadows','reflection','silver','gold',
    'glitter','confetti','sparks','dust',
    # surfaces
    'wall','door','floor','room','ceiling','corridor'
}

VISUAL_ADJECTIVES = {
    'bright','dark','soft','warm','cold','glowing','flickering','golden','silver',
    'neon','foggy','rainy','empty','lonely','dusty','broken','shattered','open',
    'closed','wide','narrow','distant','blurry','sharp','wet','dry','quiet','loud',
    'crimson','crimson','blue','red','green','yellow','black','white','pink',
    'purple','orange','gold','silver','indigo','teal','magenta'
}

def extract_visual_metaphors(lyrics: str, top_k: int = 6) -> list[str]:
    """Return concrete imageable phrases ranked by frequency."""
    text = re.sub(r'\[.*?\]', '', lyrics).lower()
    tokens = word_tokenize(text)
    tagged = pos_tag(tokens)
    candidates = Counter()

    # bigrams: ADJ + NOUN, NOUN + NOUN
    for i in range(len(tagged) - 1):
        w1, t1 = tagged[i]
        w2, t2 = tagged[i+1]
        if not (w1.isalpha() and w2.isalpha()):
            continue
        if w2 not in IMAGEABLE_NOUNS:
            continue
        if t1.startswith('JJ') and (w1 in VISUAL_ADJECTIVES or t1 == 'JJ'):
            candidates[f'{w1} {w2}'] += 2  # adj+noun is best
        elif t1.startswith('NN') and w1 in IMAGEABLE_NOUNS:
            candidates[f'{w1} {w2}'] += 1

    # solo concrete nouns get added at lower weight so we always have material
    for w, t in tagged:
        if t.startswith('NN') and w in IMAGEABLE_NOUNS:
            candidates[w] += 1

    return [c for c, _ in candidates.most_common(top_k)]

for name, lyr in TEST_LYRICS.items():
    print(f'{name:>5}: {extract_visual_metaphors(lyr)}')

## 5. Genre‑conditional anchor words + keyword extraction (Improvements #3, #6 input)

Replaces the fixed 14‑word emotion anchor list with a *per‑genre* anchor list, then merges TF‑IDF + frequency boost + visual metaphors into one ranked keyword list.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

GENRE_ANCHOR_WORDS = {
    'sad':        {'rain','shadow','tears','memory','fog','ghost','silence','alone','broken','flickering','candle','window'},
    'pop':        {'glitter','neon','confetti','golden','sun','dance','party','bright','sky','colors','sparkle','shine'},
    'rap':        {'crown','gold','chain','skyline','throne','city','king','rings','lights','stack','grind','rise'},
    'rock':       {'fire','storm','thunder','engine','leather','highway','blood','rebel'},
    'electronic': {'neon','pulse','synth','wave','circuit','glow','machine','underground'},
    'folk':       {'mountain','river','wood','lantern','field','wind','barefoot','cabin'},
}

def extract_keywords(lyrics: str, genre: str, top_k: int = 8) -> list[str]:
    tokens = preprocess(lyrics)
    raw_text = ' '.join(tokens)

    # TF-IDF treats each genre's lyrics as a doc to give us discriminative words
    corpus = [' '.join(preprocess(t)) for t in TEST_LYRICS.values()] + [raw_text]
    try:
        vec = TfidfVectorizer(max_features=40, stop_words='english', ngram_range=(1, 1))
        X = vec.fit_transform(corpus)
        feature_array = np.array(vec.get_feature_names_out())
        tfidf_scores = X.toarray()[-1]
        tfidf_top = feature_array[np.argsort(tfidf_scores)[::-1]][:12].tolist()
    except ValueError:
        tfidf_top = []

    # frequency with genre-conditional boost
    freq = Counter(tokens)
    anchors = GENRE_ANCHOR_WORDS.get(genre, set())
    for tok in list(freq.keys()):
        if tok in anchors:
            freq[tok] = int(freq[tok] * 2.5)
        elif tok in EMOTION_PRESERVE:
            freq[tok] = int(freq[tok] * 1.6)
    freq_top = [w for w, _ in freq.most_common(12)]

    # merge: visual metaphors first (most imageable), then anchored, then TF-IDF
    visual = extract_visual_metaphors(lyrics, top_k=6)
    merged = []
    seen = set()
    for src in (visual, freq_top, tfidf_top):
        for w in src:
            if w not in seen:
                merged.append(w)
                seen.add(w)
    return merged[:top_k]

for name, lyr in TEST_LYRICS.items():
    g, _ = primary_genre(lyr)
    print(f'{name:>5} (genre={g}): {extract_keywords(lyr, g)}')

## 6. Genre‑aware emotion classification (Improvements #1, #2, #5)

Three things happen here:

* The emotion model is run **once per section**, not on the full text.
* Section scores are aggregated with chorus weight 2.5× and bridge 1.5×.
* A genre‑conditional re‑weighting layer applies on top, because `j-hartmann/emotion-english-distilroberta-base` was trained on Twitter/Reddit/news. For rap, it systematically reads bravado as fear; for indie folk, it reads stoic imagery as sadness. We don't retrain — we just down‑weight known false positives and surface a `confidence` pseudo‑emotion when rap genre dominates.

In [ ]:
from transformers import pipeline

_emotion_pipe = None
def get_emotion_pipe():
    global _emotion_pipe
    if _emotion_pipe is None:
        try:
            _emotion_pipe = pipeline('text-classification',
                model='j-hartmann/emotion-english-distilroberta-base',
                top_k=None, device=0 if DEVICE == 'cuda' else -1)
        except TypeError:
            _emotion_pipe = pipeline('text-classification',
                model='j-hartmann/emotion-english-distilroberta-base',
                return_all_scores=True, device=0 if DEVICE == 'cuda' else -1)
    return _emotion_pipe

SECTION_WEIGHTS = {'chorus': 2.5, 'pre-chorus': 1.2, 'bridge': 1.5,
                   'verse': 1.0, 'verse 1': 1.0, 'verse 2': 1.0, 'intro': 0.5, 'outro': 0.5}

GENRE_EMOTION_BIAS = {
    # multipliers applied per genre after raw classification
    'rap': {'fear': 0.3, 'sadness': 0.5, 'joy': 1.2, 'anger': 1.4, 'surprise': 1.0,
            'disgust': 0.6, 'neutral': 1.0, 'confidence': 1.8},  # synthetic
    'pop': {'fear': 0.6, 'sadness': 0.5, 'joy': 1.5, 'anger': 0.5, 'surprise': 1.1,
            'disgust': 0.4, 'neutral': 0.8},
    'sad': {'fear': 1.0, 'sadness': 1.3, 'joy': 0.7, 'anger': 0.7, 'surprise': 0.8,
            'disgust': 0.8, 'neutral': 0.9},
    'rock': {'anger': 1.4, 'joy': 1.0, 'fear': 0.8},
    'electronic': {'surprise': 1.3, 'joy': 1.1, 'neutral': 1.1},
    'folk': {'sadness': 1.1, 'joy': 1.0, 'neutral': 1.2}
}

def classify_emotions(lyrics: str, genre: str) -> list[tuple[str, float]]:
    pipe = get_emotion_pipe()
    sections = split_sections(lyrics)
    accum = defaultdict(float)
    total_weight = 0.0
    for label, body in sections:
        weight = SECTION_WEIGHTS.get(label, 1.0)
        if not body.strip():
            continue
        try:
            out = pipe(body)
        except Exception as e:
            print(f'  emotion model error on section {label}: {e}')
            continue
        scores = out[0] if isinstance(out, list) and out and isinstance(out[0], list) else out
        for s in scores:
            accum[s['label']] += weight * s['score']
        total_weight += weight
    if total_weight == 0:
        return []
    raw = {k: v/total_weight for k, v in accum.items()}

    # genre-conditional re-weighting
    bias = GENRE_EMOTION_BIAS.get(genre, {})
    adjusted = {k: v * bias.get(k, 1.0) for k, v in raw.items()}
    # synthesize 'confidence' for rap-leaning content using token-level evidence
    if genre == 'rap':
        rap_signal = sum(1 for t in preprocess(lyrics) if t in GENRE_LEXICON['rap'])
        adjusted['confidence'] = min(1.0, 0.05 + 0.04 * rap_signal) * bias.get('confidence', 1.0)

    # normalize & sort
    s = sum(adjusted.values()) or 1.0
    ranked = sorted(((k, v/s) for k, v in adjusted.items()), key=lambda x: x[1], reverse=True)
    return ranked

for name, lyr in TEST_LYRICS.items():
    g, _ = primary_genre(lyr)
    em = classify_emotions(lyr, g)[:4]
    print(f'{name:>5} (genre={g}): {[(e, round(s, 3)) for e, s in em]}')

## 7. Prompt builder v5 (Improvements #6–10)

Every fix that v4 Sad needed lives here:

* **Token order** — SD reads earlier tokens with more weight, so we open with palette + lighting + framing, *then* scene, *then* style anchor.
* **Scene + lighting + color, never "cinematic" alone** — every emotion maps to a (palette, lighting, scene\_motif) triple that makes the model render colors instead of ignoring them.
* **Genre style anchor** — "in the style of a 2000s indie album cover" / "90s hip‑hop album cover" / "contemporary pop album art" gives SD a much stronger prior than "melancholic".
* **Explicit figure scale** — `subject fills 60% of frame, medium shot, centered composition` — prevents the v4 Sad "tiny figure in big empty corridor" failure.
* **Universal negative prompt** — the brown/tan output for V4 Sad would have been suppressed by `brown, tan, beige, hallway, corridor`.
* **Visual metaphors injected as the scene description** — not just emotion words.

In [ ]:
EMOTION_VISUAL = {
    'sadness': {
        'palette': 'deep cobalt blue and royal purple, indigo highlights',
        'lighting': 'soft rim lighting, single warm light source breaking through fog',
        'scene_motif': 'foggy park at dusk, lone figure under a streetlamp',
        'mood_words': 'melancholic, intimate, quiet'
    },
    'fear': {
        'palette': 'desaturated teal and charcoal, deep navy shadows',
        'lighting': 'low‑key lighting, harsh single key light, deep shadows',
        'scene_motif': 'figure looking up toward storm clouds, wind in their hair',
        'mood_words': 'tense, restrained, atmospheric'
    },
    'joy': {
        'palette': 'warm golden yellow, coral pink, sky blue, vibrant orange',
        'lighting': 'golden hour sunlight, lens flares, soft bokeh',
        'scene_motif': 'figure mid‑leap with confetti, open summer sky',
        'mood_words': 'euphoric, weightless, radiant'
    },
    'anger': {
        'palette': 'crimson red and inky black, hot ember orange',
        'lighting': 'hard backlight, sparks, harsh contrast',
        'scene_motif': 'figure facing camera, fire smoldering behind them',
        'mood_words': 'fierce, kinetic, raw'
    },
    'surprise': {
        'palette': 'electric cyan, hot magenta, pearl white',
        'lighting': 'sudden flash, high‑key, blown highlights',
        'scene_motif': 'figure with eyes wide, light bursting from behind them',
        'mood_words': 'electric, sudden, dazzling'
    },
    'disgust': {
        'palette': 'sickly green and bruise purple',
        'lighting': 'fluorescent overhead light, cold shadows',
        'scene_motif': 'figure turning away in dim alley',
        'mood_words': 'unsettled, abrasive, claustrophobic'
    },
    'neutral': {
        'palette': 'muted earth tones, soft sage and dusty rose',
        'lighting': 'overcast diffused daylight',
        'scene_motif': 'figure standing still, looking off‑frame',
        'mood_words': 'contemplative, still, observational'
    },
    'confidence': {
        'palette': 'rich gold, jet black, deep burgundy',
        'lighting': 'dramatic top light, gold hour through skyscrapers',
        'scene_motif': 'figure on a rooftop at sunset, city skyline behind',
        'mood_words': 'commanding, regal, kinetic'
    }
}

GENRE_STYLE_ANCHOR = {
    'sad':        'in the style of a 2000s indie album cover, 35mm film grain, cinematic photograph',
    'pop':        'contemporary pop album art, glossy editorial photograph, magazine cover lighting',
    'rap':        '1990s—2010s hip‑hop album cover, editorial fashion photograph, hard light',
    'rock':       'classic rock album cover, gritty 35mm film, high contrast',
    'electronic': 'modern electronic album cover, synthwave aesthetic, neon‑lit photograph',
    'folk':       'folk album cover, natural light, large format photograph'
}

GENRE_GUIDANCE_SCALE = {'sad': 8.5, 'pop': 6.5, 'rap': 7.5,
                        'rock': 7.5, 'electronic': 7.0, 'folk': 7.5}

UNIVERSAL_NEGATIVE = (
    'text, watermark, words, letters, typography, caption, subtitle, signature, '
    'logo, ugly, deformed, disfigured, low quality, blurry, jpeg artifacts, '
    'oversaturated, generic, stock photo, brown, tan, beige, sepia, dull, '
    'hallway, corridor, office building, conference room, parking lot, '
    'bad anatomy, extra limbs, malformed, low resolution'
)

@dataclass
class PromptBundle:
    prompt: str
    negative_prompt: str
    guidance_scale: float
    primary_emotion: str
    primary_genre: str
    keywords: list[str]
    concept_string: str  # ground-truth concept used for CLIP scoring

def build_prompt_v5(lyrics: str) -> PromptBundle:
    genre, _ = primary_genre(lyrics)
    keywords = extract_keywords(lyrics, genre, top_k=8)
    emotions = classify_emotions(lyrics, genre)
    primary = emotions[0][0] if emotions else 'neutral'
    visual = EMOTION_VISUAL.get(primary, EMOTION_VISUAL['neutral'])

    # build the *scene* using visual metaphors first, falling back to the canonical motif
    metaphors = [k for k in keywords if ' ' in k]
    if metaphors:
        scene = ', '.join(metaphors[:3]) + ', ' + visual['scene_motif']
    else:
        nouns = ', '.join(keywords[:4])
        scene = f"{visual['scene_motif']}, including {nouns}"

    # token order: palette → lighting → framing → scene → mood → style anchor
    prompt = (
        f"{visual['palette']}, {visual['lighting']}, "
        f"subject fills 60% of frame, medium shot, centered composition, eye‑level, "
        f"{scene}, "
        f"{visual['mood_words']}, "
        f"{GENRE_STYLE_ANCHOR.get(genre, '')}, square album cover, 1:1 aspect, highly detailed"
    )

    concept_string = (
        f"A {visual['mood_words']} {genre} album cover featuring "
        f"{', '.join(keywords[:5])} in {visual['palette']}."
    )

    return PromptBundle(
        prompt=prompt,
        negative_prompt=UNIVERSAL_NEGATIVE,
        guidance_scale=GENRE_GUIDANCE_SCALE.get(genre, 7.5),
        primary_emotion=primary,
        primary_genre=genre,
        keywords=keywords,
        concept_string=concept_string,
    )

for name, lyr in TEST_LYRICS.items():
    pb = build_prompt_v5(lyr)
    print(f'\n=== {name.upper()} → genre={pb.primary_genre} | emotion={pb.primary_emotion} | gs={pb.guidance_scale} ===')
    print(f'PROMPT: {pb.prompt}')
    print(f'CONCEPT: {pb.concept_string}')

## 8. Image generation (Improvements #11–15)

Loads SDXL when CUDA is available (1024×1024, larger CLIP encoder — the single biggest jump in color/text fidelity), else falls back to SD 1.5 with the same v5 prompts. Either way:

* `negative_prompt` is passed on every call.
* `guidance_scale` is taken from the genre table.
* `attention slicing` + `vae slicing` keep CPU/low‑VRAM stable.
* The seed is configurable per call but defaults to 42 to keep runs reproducible.
* A `controlnet_layout` hook is provided for the future improvement #14 — see the comment block at the bottom of the cell.

In [ ]:
USE_SDXL = DEVICE == 'cuda'  # SDXL on CPU is impractical (>30 min/image)
_pipe = None
_pipe_kind = None

def load_pipeline():
    """Lazy load the diffusion pipeline. SDXL on CUDA, SD 1.5 elsewhere."""
    global _pipe, _pipe_kind
    if _pipe is not None:
        return _pipe, _pipe_kind

    if USE_SDXL:
        from diffusers import StableDiffusionXLPipeline
        print('Loading SDXL base 1.0 (this is ~6.5 GB on first run)…')
        pipe = StableDiffusionXLPipeline.from_pretrained(
            'stabilityai/stable-diffusion-xl-base-1.0',
            torch_dtype=DTYPE, use_safetensors=True, variant='fp16' if DTYPE==torch.float16 else None
        )
        kind = 'sdxl'
    else:
        from diffusers import StableDiffusionPipeline
        print('Loading Stable Diffusion 1.5 (CPU mode)…')
        pipe = StableDiffusionPipeline.from_pretrained(
            'runwayml/stable-diffusion-v1-5',
            torch_dtype=DTYPE, safety_checker=None, requires_safety_checker=False
        )
        kind = 'sd15'
    pipe = pipe.to(DEVICE)
    pipe.enable_attention_slicing()
    if hasattr(pipe, 'enable_vae_slicing'):
        pipe.enable_vae_slicing()
    _pipe, _pipe_kind = pipe, kind
    print(f'✓ {kind.upper()} ready on {DEVICE}')
    return pipe, kind

def generate_image(bundle: PromptBundle, *,
                   seed: int = 42,
                   num_inference_steps: Optional[int] = None,
                   filename: Optional[str] = None) -> tuple[Image.Image, Path]:
    pipe, kind = load_pipeline()
    if num_inference_steps is None:
        num_inference_steps = 30 if kind == 'sd15' else 35
    res = 1024 if kind == 'sdxl' else 512
    g = torch.Generator(device=DEVICE).manual_seed(seed)
    t0 = time.time()
    out = pipe(
        prompt=bundle.prompt,
        negative_prompt=bundle.negative_prompt,
        guidance_scale=bundle.guidance_scale,
        num_inference_steps=num_inference_steps,
        height=res, width=res,
        generator=g,
    )
    img = out.images[0]
    elapsed = time.time() - t0
    fname = filename or f'{bundle.primary_genre}_{bundle.primary_emotion}_seed{seed}.png'
    fpath = FINAL_DIR / fname
    img.save(fpath)
    print(f'✓ Saved {fpath.name} ({elapsed:.1f}s, {res}×{res}, gs={bundle.guidance_scale}, {num_inference_steps} steps)')
    return img, fpath

# Improvement #14 (ControlNet) — wired but commented out because it adds another
# ~2GB model load and only meaningfully helps when figure placement is the failure mode.
# To enable: load `lllyasviel/control_v11p_sd15_openpose` (SD 1.5) or
# `diffusers/controlnet-canny-sdxl-1.0` (SDXL) and replace the pipeline above with
# StableDiffusionXLControlNetPipeline. Pass a generated layout (e.g. centered subject
# silhouette mask) as `image=...` to lock figure scale at 60% of frame.
print('Pipeline loader ready. Will materialize on first generate_image() call.')

## 9. CLIP‑based evaluation (Improvement #18)

CLIP gives us an objective, reproducible signal that is *independent of the prompt itself*. We score every generated image against the **concept string** (built from extracted keywords + emotion palette in plain English) so the metric measures "does the picture match what the song is about", not "did the picture follow the prompt". This avoids the v4 trap where prompt adherence was scored even when the image was emotionally wrong.

We additionally compute a **palette‑alignment score**: CLIP similarity between the image and a short list of color descriptors. That single number would have flagged V4 Sad as a failure (low blue/purple alignment, high brown alignment).

In [ ]:
from transformers import CLIPProcessor, CLIPModel

_clip = None
def get_clip():
    global _clip
    if _clip is None:
        model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(DEVICE).eval()
        proc = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
        _clip = (model, proc)
    return _clip

@torch.no_grad()
def clip_similarity(image: Image.Image, text: str) -> float:
    model, proc = get_clip()
    inputs = proc(text=[text], images=[image], return_tensors='pt', padding=True, truncation=True).to(DEVICE)
    outputs = model(**inputs)
    img_emb = outputs.image_embeds / outputs.image_embeds.norm(dim=-1, keepdim=True)
    txt_emb = outputs.text_embeds / outputs.text_embeds.norm(dim=-1, keepdim=True)
    return float((img_emb @ txt_emb.T).item())

PALETTE_TARGETS = {
    'sadness':    ['deep blue and purple photograph',  'cold tones photograph'],
    'fear':       ['dark teal and charcoal photograph','low key shadow photograph'],
    'joy':        ['bright golden warm photograph',     'sunlit colorful photograph'],
    'anger':      ['red and black fiery photograph',    'high contrast crimson photograph'],
    'surprise':   ['electric cyan and magenta photograph', 'flash high key photograph'],
    'neutral':    ['muted earth tone photograph',       'overcast pastel photograph'],
    'confidence': ['gold and black regal photograph',   'sunset rooftop photograph']
}
PALETTE_NEGATIVE = ['brown beige tan dull photograph', 'sepia stock photograph', 'corporate hallway photograph']

def palette_alignment(image: Image.Image, primary_emotion: str) -> dict:
    targets = PALETTE_TARGETS.get(primary_emotion, PALETTE_TARGETS['neutral'])
    pos = np.mean([clip_similarity(image, t) for t in targets])
    neg = np.mean([clip_similarity(image, t) for t in PALETTE_NEGATIVE])
    return {'positive': float(pos), 'negative': float(neg), 'margin': float(pos - neg)}

def evaluate_image(image: Image.Image, bundle: PromptBundle) -> dict:
    return {
        'concept_similarity': clip_similarity(image, bundle.concept_string),
        'genre_similarity':   clip_similarity(image, f'a {bundle.primary_genre} music album cover'),
        'emotion_similarity': clip_similarity(image, f'a {bundle.primary_emotion} mood photograph'),
        'palette':            palette_alignment(image, bundle.primary_emotion)
    }

print('CLIP evaluator ready.')

## 10. Multi‑rater rubric (Improvements #16, #17, #19)

The v4 report was scored by the same authors who wrote the prompts — a textbook bias source. This widget collects scores from any number of raters, blind to which prompt produced the image, on five independent axes:

| Axis | Question (1–5) |
| --- | --- |
| Emotional alignment | Does the image *feel* like the song's emotion? |
| Visual quality | Sharpness, coherence, no artifacts |
| Composition | Subject placement and framing |
| Color accuracy | Does the palette match the lyric's emotional register? |
| Album‑art viability | Would this work as actual album art? |

All scores are persisted to `final_v5/ratings.jsonl` so you can run the rating UI in multiple sessions and pool results across raters. The summary block requires ≥3 raters per image before reporting an averaged score (improvement #19).

In [ ]:
RATINGS_FILE = FINAL_DIR / 'ratings.jsonl'
RUBRIC_AXES = ['emotional_alignment', 'visual_quality', 'composition', 'color_accuracy', 'album_art_viability']

def record_rating(image_filename: str, rater_id: str, scores: dict, notes: str = ''):
    payload = {
        'image': image_filename,
        'rater_id': rater_id,
        'scores': {a: int(scores[a]) for a in RUBRIC_AXES},
        'notes': notes,
        'timestamp': time.time()
    }
    with RATINGS_FILE.open('a') as f:
        f.write(json.dumps(payload) + '\n')
    return payload

def load_ratings() -> list[dict]:
    if not RATINGS_FILE.exists():
        return []
    with RATINGS_FILE.open() as f:
        return [json.loads(l) for l in f if l.strip()]

def aggregate_ratings(min_raters: int = 3) -> dict:
    ratings = load_ratings()
    by_image = defaultdict(list)
    for r in ratings:
        by_image[r['image']].append(r)
    summary = {}
    for img, rs in by_image.items():
        if len(rs) < min_raters:
            summary[img] = {'status': f'insufficient ({len(rs)}/{min_raters})'}
            continue
        axis_scores = {a: float(np.mean([r['scores'][a] for r in rs])) for a in RUBRIC_AXES}
        axis_scores['mean']    = float(np.mean(list(axis_scores.values())))
        axis_scores['n']       = len(rs)
        summary[img] = axis_scores
    return summary

def rating_widget(image_path: Path, rater_id: str = 'anonymous'):
    """ipywidgets-based blind rating UI (no prompt shown)."""
    import ipywidgets as widgets
    from IPython.display import display, Image as IPImage, clear_output
    sliders = {a: widgets.IntSlider(value=3, min=1, max=5, description=a.replace('_', ' '),
                                    style={'description_width': '180px'},
                                    layout=widgets.Layout(width='500px')) for a in RUBRIC_AXES}
    notes  = widgets.Textarea(description='notes', layout=widgets.Layout(width='500px', height='60px'))
    submit = widgets.Button(description='submit rating', button_style='success')
    output = widgets.Output()
    def on_submit(_):
        scores = {a: s.value for a, s in sliders.items()}
        rec = record_rating(image_path.name, rater_id, scores, notes.value)
        with output:
            clear_output()
            print(f'✓ saved rating for {image_path.name} (mean={np.mean(list(scores.values())):.2f})')
    submit.on_click(on_submit)
    display(IPImage(filename=str(image_path), width=340))
    for s in sliders.values():
        display(s)
    display(notes, submit, output)

print('Rating system ready. ratings.jsonl =', RATINGS_FILE)

## 11. End‑to‑end pipeline + run on the three v4 lyric types

Putting all of this together. Each call returns a `(image, prompt_bundle, clip_metrics, filepath)` tuple and persists everything to `final_v5/`.

In [ ]:
def run_pipeline(name: str, lyrics: str, *, seed: int = 42) -> dict:
    print(f'\n{"="*72}\nPipeline run: {name}\n{"="*72}')
    bundle = build_prompt_v5(lyrics)
    print(f'→ genre={bundle.primary_genre} | emotion={bundle.primary_emotion} | guidance={bundle.guidance_scale}')
    print(f'→ keywords: {bundle.keywords}')
    print(f'→ prompt:    {bundle.prompt[:140]}…')
    img, fpath = generate_image(bundle, seed=seed, filename=f'{name}_v5.png')
    metrics = evaluate_image(img, bundle)
    print(f'→ CLIP concept_sim={metrics["concept_similarity"]:.3f}  '
          f'genre_sim={metrics["genre_similarity"]:.3f}  '
          f'palette margin={metrics["palette"]["margin"]:+.3f}')
    record = {
        'name': name,
        'bundle': asdict(bundle),
        'metrics': metrics,
        'filepath': str(fpath),
        'seed': seed,
    }
    with (FINAL_DIR / f'{name}_run.json').open('w') as f:
        json.dump(record, f, indent=2)
    return record

# Heads-up: on CPU each image takes ~10–15 minutes. Run one at a time if you want
# to spot-check results, or run all three and walk away.
RESULTS = {}
for nm, lyr in TEST_LYRICS.items():
    RESULTS[nm] = run_pipeline(nm, lyr)

## 12. Side‑by‑side: V4 (sad) vs new V5

Specifically targets the failure case from the report — V4 Sad's brown corridor with the tiny figure. Reads `improved_main`'s `sad_v4.png` (if present) and the new `sad_v5.png`, and shows them next to each other along with their CLIP/palette scores. If the old V4 file isn't on disk we just show the new one.

In [ ]:
import matplotlib.pyplot as plt

def show_comparison(name: str):
    candidates_old = [
        OUTPUT_DIR / f'{name}_v4.png',
        OUTPUT_DIR / f'{name}_4.png',
        OUTPUT_DIR / f'sad_introspective_v4.png' if name == 'sad' else None,
    ]
    old_path = next((p for p in candidates_old if p and p.exists()), None)
    new_path = FINAL_DIR / f'{name}_v5.png'
    if not new_path.exists():
        print(f'(no new image for {name} yet — run section 11)')
        return
    n = 2 if old_path else 1
    fig, axes = plt.subplots(1, n, figsize=(6*n, 6))
    if n == 1:
        axes = [axes]
    if old_path:
        axes[0].imshow(Image.open(old_path)); axes[0].set_title(f'V4 (old): {old_path.name}'); axes[0].axis('off')
        axes[1].imshow(Image.open(new_path)); axes[1].set_title(f'V5 (new): {new_path.name}'); axes[1].axis('off')
    else:
        axes[0].imshow(Image.open(new_path)); axes[0].set_title(f'V5: {new_path.name}'); axes[0].axis('off')
    plt.tight_layout(); plt.show()
    if name in RESULTS:
        m = RESULTS[name]['metrics']
        print(f"V5 metrics: concept={m['concept_similarity']:.3f}  "
              f"genre={m['genre_similarity']:.3f}  emotion={m['emotion_similarity']:.3f}  "
              f"palette +ve={m['palette']['positive']:.3f}  -ve={m['palette']['negative']:.3f}  "
              f"margin={m['palette']['margin']:+.3f}")

for nm in TEST_LYRICS:
    print('\n' + '='*72); print(nm.upper()); print('='*72)
    show_comparison(nm)

## 13. Ablations

Three quick ablations to demonstrate each upgrade's contribution. Each varies one component while holding the other four constant. Useful for the next report — the v4 paper had no ablations.

In [ ]:
ABLATION_LYRICS = TEST_LYRICS['sad']

ABLATIONS = {
    'A_no_visual_metaphors': {
        'description': 'Strip visual-metaphor extraction — use only emotion words.',
        'mutate': lambda b: PromptBundle(
            prompt=re.sub(r'foggy park.*?streetlamp', 'shadows, cry, night', b.prompt),
            negative_prompt=b.negative_prompt, guidance_scale=b.guidance_scale,
            primary_emotion=b.primary_emotion, primary_genre=b.primary_genre,
            keywords=b.keywords, concept_string=b.concept_string)
    },
    'B_no_negative_prompt': {
        'description': 'Drop the universal negative prompt entirely.',
        'mutate': lambda b: PromptBundle(
            prompt=b.prompt, negative_prompt='', guidance_scale=b.guidance_scale,
            primary_emotion=b.primary_emotion, primary_genre=b.primary_genre,
            keywords=b.keywords, concept_string=b.concept_string)
    },
    'C_palette_last': {
        'description': 'Move palette/lighting tokens to the END of the prompt (mimics V4 ordering).',
        'mutate': lambda b: PromptBundle(
            prompt=re.sub(r'^(.*?), (subject fills.*)$',
                          r'\2, \1', b.prompt, count=1) if 'subject fills' in b.prompt else b.prompt,
            negative_prompt=b.negative_prompt, guidance_scale=b.guidance_scale,
            primary_emotion=b.primary_emotion, primary_genre=b.primary_genre,
            keywords=b.keywords, concept_string=b.concept_string)
    },
}

def run_ablations(lyrics: str, ablations: dict):
    base = build_prompt_v5(lyrics)
    rows = [('full_v5', base)]
    for code, spec in ablations.items():
        rows.append((code, spec['mutate'](base)))
    out = []
    for code, b in rows:
        img, fp = generate_image(b, filename=f'sad_ablation_{code}.png', seed=42)
        m = evaluate_image(img, b)
        out.append({'code': code, 'desc': ABLATIONS.get(code, {}).get('description', 'baseline'),
                    'metrics': m, 'filepath': str(fp)})
    return out

# Uncomment to run — each ablation = one full image generation.
# ABL_RESULTS = run_ablations(ABLATION_LYRICS, ABLATIONS)
# print(json.dumps(ABL_RESULTS, indent=2, default=str))
print('Ablation harness loaded. Uncomment the run_ablations call when ready.')

## 14. Collecting blind rater scores

Hand the laptop to a rater (or rate yourself per‑image), then run `aggregate_ratings()` after ≥3 raters have submitted scores.

```python
rating_widget(FINAL_DIR / 'sad_v5.png', rater_id='rater_1')
rating_widget(FINAL_DIR / 'sad_v5.png', rater_id='rater_2')
rating_widget(FINAL_DIR / 'sad_v5.png', rater_id='rater_3')
aggregate_ratings(min_raters=3)
```

In [ ]:
# Example: launch a rating widget for the first generated image.
# In practice you'd run this cell once per (image, rater) pair.
first_img = next(iter(FINAL_DIR.glob('*_v5.png')), None)
if first_img is not None:
    print(f'Launching widget for {first_img.name}…')
    rating_widget(first_img, rater_id='vikram')
else:
    print('Run section 11 first to generate images.')

In [ ]:
summary = aggregate_ratings(min_raters=3)
print(json.dumps(summary, indent=2))

## 15. Where this notebook still falls short

Honest accounting of what's wired but not fully solved:

* **Improvement #5 (lyrics‑specific fine‑tuned model)** — we use the genre‑aware re‑weighting of `j-hartmann/...` as a stand‑in. A real fix is to fine‑tune DistilRoBERTa on the WASABI lyrics corpus or the `MoodyLyrics` dataset.
* **Improvement #14 (ControlNet)** — the loader stub is documented but not active by default. Wiring it requires a second 1.5–2 GB model download and a layout image generator. Worth doing on a GPU machine; on CPU it more than doubles per‑image time without a proportional quality gain.
* **Genre lexicon classifier** — deterministic and transparent, but a real DistilBERT genre head trained on WASABI would generalize better to ironic / cross‑genre songs.
* **CLIP evaluation** — CLIP itself has biases (Western art, English captions). Two CLIP models (CLIP‑L/14 + SigLIP) averaged would be more robust than ours.
* **Rater pool** — with three friends as raters we still have a small‑sample bias. The published version of this work should target n≥15 raters drawn outside the project team.

What it *does* fix relative to V4 Sad: the palette is no longer ignored (palette + lighting tokens are first), the figure is forced to ~60% of the frame, the universal negative prompt suppresses the brown/corridor failure mode, and the foggy‑park motif from V1 Sad (the emotionally strongest output of the project) is now baked into the sadness scene template instead of being lost when prompt verbosity ballooned in V4.